# Optimized MH MCMC algortihm

In the Std_mcmc.ipynb code, appending rows using mcmc_df.loc[len(mcmc_df)] = ... inside a loop can significantly slow down execution for larger iteration counts ($N > 10,000$). For optimal performance, append to a standard Python list during the loop and convert to a pd.DataFrame at the end:

In [4]:
import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

c = 299792.458 

data = pd.read_csv("Pantheon_SNdata.csv") # To check the data, you can use: print(data.head())

---



In [13]:
def chi_sq(obs, theo, err):
    obs= np.array(obs)
    theo=np.array(theo)
    err=np.array(err)
    chi_squared=np.sum(((obs-theo)/err)**2)
    return chi_squared

def plot_hist_with_gaussian(data, param_name, bins=50, color='#7F77DD'):
    """Histogram of MCMC samples with a fitted Gaussian overlay."""
    data = np.asarray(data)
 
    # Fit a Gaussian: returns (mean, std) that best match the sample distribution
    mu, sigma = stats.norm.fit(data)
 
    fig, ax = plt.subplots(figsize=(6, 4))
 
    # density=True normalizes the histogram so its area = 1, matching the
    # Gaussian pdf's area — otherwise the curve and bars are on different scales
    counts, bin_edges, _ = ax.hist(data, bins=bins, density=True,
                                    color=color, alpha=0.6,
                                    edgecolor='white', linewidth=0.3,
                                    label='MCMC samples')
 
    x = np.linspace(data.min(), data.max(), 300)
    pdf = stats.norm.pdf(x, mu, sigma)
    ax.plot(x, pdf, color='black', lw=2,
            label=f'Gaussian fit\n$\\mu$={mu:.7f}, $\\sigma$={sigma:.7f}')
 
    ax.axvline(mu, color='red', linestyle='--', lw=0.8)
 
    ax.set_xlabel(param_name)
    ax.set_ylabel('Probability density')
    ax.set_title(f'Posterior distribution: {param_name}')
    ax.legend()
    plt.tight_layout()
    plt.show()
 
    return mu, sigma

---


In [6]:
def Hubble(z, H0, omega_m, w):
    return H0 * np.sqrt(omega_m * (1 + z)**3 + (1 - omega_m) * (1 + z)**(3 * (1 + w)))

def integrand(z, H0, omega_m, w):
    return c / Hubble(z, H0, omega_m, w)

def DISTANCE_MODULUS(z, H0, omega_m, w):
    D_c, error = integrate.quad(integrand, 0, z, args=(H0, omega_m, w))
    D_L = D_c * (1 + z)
    return 5*np.log10(D_L) + 25

def CALCULATE_NEW(omega_m_prev, H0_prev, w_prev,
                    domega_m, dH0, dw,
                    omega_m_range, H0_range, w_range):
    omega_m_new = omega_m_prev + domega_m * np.random.normal(0, 1)
    H0_new = H0_prev + dH0 * np.random.normal(0, 1)
    w_new = w_prev + dw * np.random.normal(0, 1)
    if (omega_m_range[0] <= omega_m_new <= omega_m_range[1]
        and H0_range[0] <= H0_new <= H0_range[1]
        and w_range[0] <= w_new <= w_range[1]):
        return omega_m_new, H0_new, w_new
    return CALCULATE_NEW(omega_m_prev, H0_prev, w_prev, domega_m, dH0, dw,
                         omega_m_range, H0_range, w_range)

In [ ]:

# define some constants (parameters)
omega_m_range = [0.15, 0.45]
H0_range = [55, 85]
w_range = [-2.0, 0.0]
delta_w = 0.03
delta_omega_m = 0.005
delta_H0 = 0.2
print(f'\u03A9_m range:{omega_m_range}, \u0394\u03A9_m:{delta_omega_m},\nH0 range:{H0_range}, \u0394H0:{delta_H0},\nw range:{w_range}, \u0394w:{delta_w}')

N_iterations =10000 

Ω_m range:[0.15, 0.45], ΔΩ_m:0.005,
H0 range:[55, 85], ΔH0:0.2,
w range:[-2.0, 0.0], Δw:0.03


In [ ]:
omega_m = np.random.uniform(omega_m_range[0], omega_m_range[1])
H0 = np.random.uniform(H0_range[0], H0_range[1])
w = np.random.uniform(w_range[0], w_range[1])

chi_sq1 = chi_sq(data['dist_mod'], [DISTANCE_MODULUS(z, H0, omega_m, w) for z in data['zcmb']], data['dmb'])

# Create an empty list to store chain entries
chain_records = []

for n in range(N_iterations):
    omega_m_new, H0_new, w_new = CALCULATE_NEW(omega_m, H0, w, delta_omega_m, delta_H0, delta_w, omega_m_range, H0_range, w_range)
    
    chi_sq2 = chi_sq(data['dist_mod'],[DISTANCE_MODULUS(z, H0_new, omega_m_new, w_new) for z in data['zcmb']],data['dmb'])

    if chi_sq2 <= chi_sq1: # Type 1 Acceptance
        omega_m, H0, w, chi_sq1 = omega_m_new, H0_new, w_new, chi_sq2
        chain_records.append({'Omega_m': omega_m, 'H0': H0, 'w': w, 'Chi_sq': chi_sq1, 'Type': 1})
        
    else:
        nrand = np.random.uniform(0, 1)
        prob_ratio = np.exp(-(chi_sq2 - chi_sq1) / 2)
        
        if prob_ratio > nrand: # Type 2 Acceptance
            omega_m, H0, w, chi_sq1 = omega_m_new, H0_new, w_new, chi_sq2
            chain_records.append({'Omega_m': omega_m, 'H0': H0, 'w': w, 'Chi_sq': chi_sq1, 'Type': 2})
            
        else: # Type 3 Rejection
            pass


mcmc_df = pd.DataFrame(chain_records)
mcmc_df.to_csv('Saved Data\\mcmc_cosmology_chain.csv', index=False)